<a href="https://colab.research.google.com/github/prateek-sahu/Discovery-of-Handwashing/blob/master/Classification_Agent_With_MCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Read from .env later
import os
os.environ["OPENAI_API_KEY"]  = ""
os.environ["LANGFUSE_PUBLIC_KEY"]  = ""
os.environ["LANGFUSE_SECRET_KEY"]  = ""
os.environ["LANGFUSE_HOST"]        = "https://cloud.langfuse.com"


In [2]:
from langfuse import Langfuse
langfuse = Langfuse()

# Run the test
langfuse.auth_check()
print("Success! Your keys are working.")


Success! Your keys are working.


In [3]:


## Step 1 — Install Packages
!pip install -q langchain-openai langchain-mcp-adapters langfuse fastmcp pandas
## Step 2 — Import Packages
#pip install fastmcp[server]
import os
import threading
import time
import pandas as pd

from fastmcp import FastMCP
#from langchain_openai import AzureChatOpenAI
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage, SystemMessage
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

print("All packages imported successfully ✅")
## Step 3 — Configure Environment Variables

# ── Azure OpenAI ──────────────────────────────────────────────

#os.environ["AZURE_OPENAI_ENDPOINT"] = "https://YOUR_RESOURCE.openai.azure.com/"
#AZURE_DEPLOYMENT  = "gpt-4o"          # your deployment name
#AZURE_API_VERSION = "2024-02-01"

# ── Langfuse ──────────────────────────────────────────────────


print("Environment configured ✅")
## Step 4 — Initialize LLM & Langfuse
# Langfuse client (for dataset management & scoring)
langfuse = Langfuse()

# Langfuse LangChain callback handler (auto-traces every LLM call)
langfuse_handler = CallbackHandler()


#  OpenAI LLM
llm = ChatOpenAI(
    model = "gpt-4o",
    temperature=0,
    callbacks=[langfuse_handler]   # attach tracing
)

print("LLM and Langfuse initialised ✅")
## Step 5 — Create MCP Server with FastMCP

mcp = FastMCP("ClassificationAgentServer")
print("MCP server object created ✅")

## Step 6 — Register MCP Tools
@mcp.tool()
def calculator(expression: str) -> str:
    """
    Evaluate a safe mathematical expression and return the result.
    Input : arithmetic string e.g. '45 + 90' or '3 * (10 / 2)'
    Output: string representation of the computed value.
    """
    try:
        # Restrict to safe tokens only (no builtins, no imports)
        allowed = set("0123456789 +-*/().")
        if not all(c in allowed for c in expression):
            return "Error: only numeric arithmetic expressions are allowed."
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {e}"

print("calculator tool registered ✅")
@mcp.tool()
def summary(text: str) -> str:
    """
    Summarise a long block of text into a concise paragraph.
    Input : raw text (any length)
    Output: 2-3 sentence summary.
    """
    messages = [
        SystemMessage(content="You are an expert summariser. Return a 2-3 sentence summary, preserving key facts."),
        HumanMessage(content=f"Summarise the following:\n\n{text}")
    ]
    response = llm.invoke(messages)
    return response.content

print("summary tool registered ✅")
@mcp.tool()
def info(topic: str) -> str:
    """
    Return a clear, factual explanation of a technical topic.
    Input : topic name e.g. 'Apache Kafka', 'Docker', 'Kubernetes'
    Output: structured explanation (what it is, why it matters, example use-case).
    """
    messages = [
        SystemMessage(content=(
            "You are a senior software engineer. "
            "Explain the topic in 3 short sections: "
            "(1) What it is, (2) Why it matters, (3) One real-world use-case. "
            "Keep the response under 150 words."
        )),
        HumanMessage(content=f"Explain: {topic}")
    ]
    response = llm.invoke(messages)
    return response.content

print("info tool registered ✅")
## Step 7 — Start MCP Server (background thread)
def run_server():
    """Run FastMCP with SSE transport on port 8001 (blocking)."""
    mcp.run(transport="sse", host="127.0.0.1", port=8001)

# Daemon thread: terminates automatically when the notebook kernel stops
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give the server a moment to bind the port
time.sleep(2)
print("MCP server running at http://127.0.0.1:8001/mcp ✅")
## Step 8 — Connect MCP Client
!pip install nest_asyncio
import asyncio
import nest_asyncio

nest_asyncio.apply()

"""Connect to the MCP server and retrieve registered tools."""
mcp_client= MultiServerMCPClient(
        {
            "classification_agent": {
                "url": "http://127.0.0.1:8001/sse",
                "transport": "sse",
            }
        }
    )

async def get_tools():
    tools = await mcp_client.get_tools()
    return tools

# Run in notebook event loop
mcp_tools = asyncio.run(get_tools())
print(f"Connected to MCP. Tools available: {[t.name for t in mcp_tools]} ✅")
## Step 9 — Create Test Dataset
test_data = [
    # ── Informational ───────────────────────────────────────────
    {"query": "What is Apache Kafka?",                                      "expected": "info"},
    {"query": "Explain what Docker containers are.",                        "expected": "info"},
    {"query": "What is Kubernetes used for?",                               "expected": "info"},
    {"query": "Tell me about LangChain.",                                   "expected": "info"},

    # ── Action / Calculation ────────────────────────────────────
    {"query": "Calculate 500 + 700",                                        "expected": "action"},
    {"query": "What is 15 * 8?",                                            "expected": "action"},
    {"query": "Compute (100 / 4) + 25",                                     "expected": "action"},
    {"query": "Evaluate 99 - 43",                                           "expected": "action"},

    # ── Summary ─────────────────────────────────────────────────
    {"query": "Summarize this article about climate change: Global temperatures have risen by 1.1°C since pre-industrial times. Ice caps are melting. Sea levels are rising. Extreme weather events are increasing in frequency.",
     "expected": "summary"},
    {"query": "Give me a short summary of this paragraph: Machine learning is a branch of AI that enables systems to learn from data without being explicitly programmed. It relies on algorithms that iteratively learn from data to improve their predictions.",
     "expected": "summary"},
    {"query": "Can you summarize the following text: The Python programming language was created by Guido van Rossum and first released in 1991. It emphasises code readability and simplicity.",
     "expected": "summary"},
    {"query": "TL;DR this: The Agile methodology is an iterative approach to project management. Teams complete work in sprints, review results, and adapt the plan accordingly.",
     "expected": "summary"},
]

df = pd.DataFrame(test_data)
print(f"Dataset created: {len(df)} rows")
df.head()
## Step 10 — Intent Classifier V1 (Minimal Prompt)
# ── V1: bare-minimum prompt ──────────────────────────────────────
SYSTEM_PROMPT_V1 = """Return ONLY one word: info OR action OR summary."""

def classify_query_v1(query: str) -> str:
    """
    Classify a user query using the minimal V1 prompt.
    Returns: 'info', 'action', or 'summary'
    """
    messages = [
        SystemMessage(content=SYSTEM_PROMPT_V1),
        HumanMessage(content=query)
    ]
    response = llm.invoke(messages)          # traced by langfuse_handler
    return response.content.strip().lower()

# Quick smoke-test
print(classify_query_v1("What is Docker?"))      # expected: info
print(classify_query_v1("Calculate 10 + 5"))     # expected: action
print(classify_query_v1("Summarize this text"))  # expected: summary
# Run V1 predictions on the full dataset
df["predicted_v1"] = df["query"].apply(classify_query_v1)

# Accuracy
v1_accuracy = (df["predicted_v1"] == df["expected"]).mean()
print(f"\nV1 Accuracy: {v1_accuracy:.0%}")

# Wrong predictions
wrong_v1 = df[df["predicted_v1"] != df["expected"]]
print(f"\nWrong predictions ({len(wrong_v1)} rows):")
print(wrong_v1[["query", "expected", "predicted_v1"]].to_string(index=False))
## Step 11 — Observe Langfuse Traces
'''
Open [cloud.langfuse.com](https://cloud.langfuse.com) → **Traces** to see:
- Every LLM call with input/output
- Token usage & latency per call
- Which queries were misclassified

**Common V1 failure patterns to look for:**
- Ambiguous queries returning extra words beyond the label
- Math queries with natural language preamble classified as `info`
- No examples → model guesses intent from syntax alone
'''
## Step 12 — Build Routing Agent V1
async def assistant_v1(query: str) -> str:
    """
    Full routing agent:
      1. Classify intent
      2. Call the matching MCP tool
      3. Return the tool response
    """
    # ── Step 1: classify ──────────────────────────────────────────
    intent = classify_query_v1(query)
    print(f"  [Router] intent detected → '{intent}'")

    # ── Step 2: route to MCP tool ─────────────────────────────────

    tools = await mcp_client.get_tools

    if intent == "action":
        # Extract the expression (strip natural language wrapper)
        expr = query.replace("Calculate", "").replace("Compute", "") \
                    .replace("Evaluate", "").replace("What is", "") \
                    .replace("?", "").strip()
        result = await tools["calculator"].ainvoke({"expression": expr})

    elif intent == "summary":
        # Pass full query text (the tool will summarise it)
        result = await tools["summary"].ainvoke({"text": query})

    else:  # info (default)
        result = await tools["info"].ainvoke({"topic": query})

    return result

print("Routing agent defined ✅")
async def assistant_v1(query: str) -> str:
    # Step 1: classify intent
    intent = classify_query_v1(query)
    print(f"  [Router] intent detected → '{intent}'")

    # Step 2: get tools
    tools = await mcp_client.get_tools()

    # Step 3: route based on intent
    if intent == "action":
        calc_tool = next((t for t in tools if t.name == "calculator"), None)
        if calc_tool:
            expr = query.replace("Calculate", "").replace("Compute", "") \
                        .replace("Evaluate", "").replace("What is", "") \
                        .replace("?", "").strip()
            result = await calc_tool.ainvoke({"expression": expr})
            return str(result)

    elif intent == "summary":
        sum_tool = next((t for t in tools if t.name == "summary"), None)
        if sum_tool:
            result = await sum_tool.ainvoke({"text": query})
            return str(result)

    else:  # info
        info_tool = next((t for t in tools if t.name == "info"), None)
        if info_tool:
            result = await info_tool.ainvoke({"topic": query})
            return str(result)

    return "Could not route query."


# ── Run tests using await directly (not asyncio.run) ──────────
async def run_all_tests():
    print("--- Test 1: Info ---")
    print(await assistant_v1("What is Apache Kafka?"))
    print("\n--- Test 2: Action ---")
    print(await assistant_v1("Calculate 500 + 700"))
    print("\n--- Test 3: Summary ---")
    print(await assistant_v1(
        "Summarize this: Machine learning is a branch of AI "
        "that enables systems to learn from data."
    ))



All packages imported successfully ✅
Environment configured ✅
LLM and Langfuse initialised ✅
MCP server object created ✅
calculator tool registered ✅
summary tool registered ✅
info tool registered ✅


╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.4.2                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │               🖥  Server:      ClassificationAgentServer, 3.4.2               │                  
                 │               🚀 Deploy free: https://horizon.prefect.io                     │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[06/23/26 16:47:33] INFO     Starting MCP server 'ClassificationAgentServer' with transport 'sse'  ]8;id=19145;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=189372;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#304\304]8;;\
                             on http://127.0.0.1:8001/sse                                                          

INFO:     Started server process [7789]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


MCP server running at http://127.0.0.1:8001/mcp ✅
INFO:     127.0.0.1:41402 - "GET /sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:41404 - "POST /messages/?session_id=b995371ea7234e219420c085a4f0456d HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:41404 - "POST /messages/?session_id=b995371ea7234e219420c085a4f0456d HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:41404 - "POST /messages/?session_id=b995371ea7234e219420c085a4f0456d HTTP/1.1" 202 Accepted
Connected to MCP. Tools available: ['calculator', 'summary', 'info'] ✅
Dataset created: 12 rows
info
action
summary

V1 Accuracy: 100%

Wrong predictions (0 rows):
Empty DataFrame
Columns: [query, expected, predicted_v1]
Index: []
Routing agent defined ✅


In [4]:
asyncio.run(run_all_tests())
## Step 13 — Test the Agent
# Test 3: Summary query
long_text = (
    "Summarize this paragraph: "
    "Machine learning is a branch of artificial intelligence that gives computers "
    "the ability to learn without being explicitly programmed. It focuses on the "
    "development of algorithms that can access data and use it to learn for themselves. "
    "The process begins with observations or data, such as examples, direct experience, "
    "or instruction, to look for patterns in data and make better decisions in the future."
)
print("=" * 60)
print("QUERY: (summary of ML paragraph)")
result = asyncio.run(assistant_v1(long_text))
print("RESPONSE:")
print(result)
## Step 14 — Improve Prompt (V2)
'''
V1 weaknesses identified:
- No definitions → model guesses
- No examples → ambiguous queries fail
- No output format enforcement → occasionally returns extra words

V2 fixes all of these with few-shot examples, explicit definitions, and strict output constraints.
'''
SYSTEM_PROMPT_V2 = """\
You are an intent classification engine. Your ONLY job is to output exactly one of these three labels:
  info | action | summary

## Definitions
- **info**   : The user wants a factual explanation or description of a concept, tool, or technology.
- **action** : The user wants a computation, calculation, or arithmetic result.
- **summary**: The user wants a long piece of text condensed into a shorter form.

## Few-shot Examples
| Query                                          | Label   |
|------------------------------------------------|---------|
| What is Apache Kafka?                          | info    |
| Explain what Docker containers are.            | info    |
| Calculate 45 + 90                              | action  |
| What is 15 * 8?                                | action  |
| Evaluate (100 / 4) + 25                        | action  |
| Summarize this article: ...                    | summary |
| Give me a short summary of this paragraph: ... | summary |
| TL;DR this: ...                                | summary |

## Output Rules
- Output EXACTLY ONE WORD: info, action, or summary.
- No punctuation, no explanation, no extra text.
- If unsure between action and info, prefer action when numbers are present.
- If unsure between summary and info, prefer summary when the query contains a block of text to process.
"""

def classify_query_v2(query: str) -> str:
    """
    Classify a user query using the improved V2 prompt.
    Returns: 'info', 'action', or 'summary'
    """
    messages = [
        SystemMessage(content=SYSTEM_PROMPT_V2),
        HumanMessage(content=query)
    ]
    response = llm.invoke(messages)
    return response.content.strip().lower()

# Smoke test
print(classify_query_v2("What is Docker?"))           # expected: info
print(classify_query_v2("Calculate (50 * 2) + 10"))   # expected: action
print(classify_query_v2("TL;DR this long article"))   # expected: summary
# Run V2 predictions on the full dataset
df["predicted_v2"] = df["query"].apply(classify_query_v2)

# Accuracy
v2_accuracy = (df["predicted_v2"] == df["expected"]).mean()
print(f"V2 Accuracy: {v2_accuracy:.0%}")

wrong_v2 = df[df["predicted_v2"] != df["expected"]]
print(f"\nWrong predictions ({len(wrong_v2)} rows):")
if len(wrong_v2) == 0:
    print("None — perfect classification!")
else:
    print(wrong_v2[["query", "expected", "predicted_v2"]].to_string(index=False))
# ── V1 vs V2 Comparison ─────────────────────────────────────────────
comparison = pd.DataFrame({
    "Metric":   ["Accuracy", "Wrong Predictions"],
    "V1 (minimal prompt)": [
        f"{v1_accuracy:.0%}",
        str(len(df[df['predicted_v1'] != df['expected']]))
    ],
    "V2 (few-shot + definitions)": [
        f"{v2_accuracy:.0%}",
        str(len(df[df['predicted_v2'] != df['expected']]))
    ],
})
print(comparison.to_string(index=False))

# Per-row diff
print("\nPer-query prediction table:")
print(df[["query", "expected", "predicted_v1", "predicted_v2"]].to_string(index=False))
## V1 vs V2 — Improvement Reasoning



--- Test 1: Info ---
  [Router] intent detected → 'info'
INFO:     127.0.0.1:58608 - "GET /sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:58624 - "POST /messages/?session_id=836f7b2d7a004dcb99de555feb1ca3b8 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58624 - "POST /messages/?session_id=836f7b2d7a004dcb99de555feb1ca3b8 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58624 - "POST /messages/?session_id=836f7b2d7a004dcb99de555feb1ca3b8 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58634 - "GET /sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:58636 - "POST /messages/?session_id=4255b0c087844e55868e5c2eb73816b3 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58636 - "POST /messages/?session_id=4255b0c087844e55868e5c2eb73816b3 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58636 - "POST /messages/?session_id=4255b0c087844e55868e5c2eb73816b3 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58636 - "POST /messages/?session_id=4255b0c087844e55868e5c2eb73816b3 HTTP/1.1" 202 Accepted
[{'type': 'text', 'text': '(1) **What it is:** Ap